# SLD-VM: Fine-Tuned Gemma 2B + Pipeline
## Target: 34–38% on GSM8K

**Strategy:**
```
Phase 1 (~4 hrs): LoRA fine-tune Gemma 2B on GSM8K train set (7,473 problems)
Phase 2 (~2 hrs): Run SLD-VM pipeline on fine-tuned model
```

**Why this works:**
- Fine-tuning teaches the model HOW to do structured math reasoning (puts capability in weights)
- SLD-VM pipeline then amplifies that capability via self-consistency and verification
- Fine-tuned 2B alone: ~28-32% | Fine-tuned 2B + pipeline: ~34-38%

**Architecture:**
```
GSM8K Train (7473 problems)
    ↓ LoRA SFT (3 epochs)
Fine-tuned Gemma 2B
    ↓ Structured CoT (temp=0)
    ↓ Self-Consistency (3 branches)
    ↓ VCE (Proxy PRM + arithmetic)
    → Final Answer
```

**Runtime on P100/T4:**
- Fine-tuning: ~3-5 hours
- Evaluation (200 problems): ~2-3 hours
- Total: ~6-8 hours (plan for an overnight run)

---
# PHASE 1: FINE-TUNING
---

## Cell 1 — Install Dependencies

In [1]:
# Install fine-tuning stack
# trl: training library with SFTTrainer
# peft: LoRA adapter library
# bitsandbytes: 4-bit quantization (optional but saves VRAM)
# !pip install -q trl peft bitsandbytes accelerate transformers datasets
!pip install bitsandbytes

print('✅ Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01
✅ Dependencies installed.


## Cell 2 — Login & Configuration

In [3]:
from huggingface_hub import login
login("")
print('✅ HuggingFace login successful.')

✅ HuggingFace login successful.


In [30]:
import os
import torch
import numpy as np

# ═══════════════════════════════════════════════════════════
# CONFIGURATION — Read carefully before running
# ═══════════════════════════════════════════════════════════

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'
DEVICE = 'cuda'
RANDOM_SEED = 42

# Fine-tuning config
FINETUNE_CONFIG = {
    'num_train_epochs': 3,         # 3 epochs is the sweet spot for GSM8K
                                   # Less = underfitting, More = overfitting
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,  # Effective batch size = 2*8 = 16 
    'learning_rate': 2e-4,         # Standard LoRA LR
    'warmup_ratio': 0.05,
    'max_seq_length': 512,         # GSM8K solutions rarely exceed 400 tokens
    'save_steps': 100,
    'logging_steps': 20,
}

# LoRA config
LORA_CONFIG = {
    'r': 32,                       # Rank — higher = more parameters, slower, smarter
                                   # 16 is the standard for reasoning tasks
    'lora_alpha': 32,              # Scale = alpha/r = 2.0 (standard)
    'lora_dropout': 0.05,
    # Target all linear projection layers in Gemma
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    'bias': 'none',
    'task_type': 'CAUSAL_LM',
}

# Evaluation config (applied AFTER fine-tuning)
EVAL_CONFIG = {
    'n_eval': 200,                 # Dev set — change to 1319 for full paper results
    'n_branches': 3,
    'branch_temperatures': [0.3, 0.6, 0.9],
    'max_new_tokens': 512,
    'vce_prm_weight': 0.7,
    'vce_rule_weight': 0.3,
}

# Output directories
FINETUNED_MODEL_DIR = '/kaggle/working/gemma2b_gsm8k_lora/'
EVAL_OUTPUT_DIR = '/kaggle/working/sldvm_finetuned_eval/'
os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

# Seeds
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Configuration set.')
print(f'Model : {MODEL_ID}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Fine-tuned model will be saved to: {FINETUNED_MODEL_DIR}')

✅ Configuration set.
Model : microsoft/Phi-3-mini-4k-instruct
GPU: Tesla T4
VRAM: 15.6 GB
Fine-tuned model will be saved to: /kaggle/working/gemma2b_gsm8k_lora/


## Cell 3 — Prepare Training Data

GSM8K training answers contain `<<arithmetic>>` calculator annotations like `<<16-3-4=9>>`. We strip these to get clean natural-language reasoning chains that the model can learn from.

Each training example is formatted as:
```
What I need to find: [goal]
Step 1: [reasoning]
Step 2: [reasoning]
...
Answer: [number]
```

This matches exactly the format our SLD-VM pipeline will prompt for at inference time — the model learns to produce exactly the format we'll consume.

In [8]:
import re
from datasets import load_dataset

print('Loading GSM8K train split...')
gsm8k_train = load_dataset('gsm8k', 'main', split='train')
gsm8k_test = load_dataset('gsm8k', 'main', split='test')

print(f'Train size: {len(gsm8k_train)}')
print(f'Test size:  {len(gsm8k_test)}')
print(f'\nRaw training example:')
print(f'Q: {gsm8k_train[0]["question"]}')
print(f'A: {gsm8k_train[0]["answer"]}')

Loading GSM8K train split...
Train size: 7473
Test size:  1319

Raw training example:
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
A: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [9]:
def clean_gsm8k_solution(raw_answer):
    """
    Convert raw GSM8K answer into clean CoT format.

    Input:  'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs...\n#### 18'
    Output: Clean step-by-step solution ending with 'Answer: 18'
    """
    # Strip calculator annotations: <<16-3-4=9>> → empty
    clean = re.sub(r'<<[^>]+>>', '', raw_answer)

    # Extract final answer
    final_match = re.search(r'####\s*([\-0-9,\.]+)', clean)
    final_answer = final_match.group(1).replace(',', '').strip() if final_match else ''

    # Remove the #### line from body
    body = re.sub(r'####.*', '', clean).strip()

    # Split into sentences and number them as steps
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', body) if s.strip()]

    # Build structured steps
    steps = []
    for i, sent in enumerate(sentences, 1):
        if sent:
            steps.append(f'Step {i}: {sent}')

    solution = '\n'.join(steps)
    if final_answer:
        solution += f'\nAnswer: {final_answer}'

    return solution


def format_training_example(example):
    """
    Format a GSM8K example into the instruction-following format
    that matches our SLD-VM inference prompt exactly.
    """
    problem = example['question']
    solution = clean_gsm8k_solution(example['answer'])

    # This prompt format MUST match STRUCTURED_COT_PROMPT used at inference
    # The model will learn to complete this exact template
    prompt = f"""Solve this math problem carefully.

First, identify what the problem is asking for.
Then solve step by step, tracking all intermediate values.
Finally, write your answer.

Problem: {problem}

What I need to find: {solution}"""

    return {'text': prompt}


# Apply formatting
print('Formatting training data...')
train_dataset = gsm8k_train.map(format_training_example, remove_columns=gsm8k_train.column_names)

print(f'✅ Formatted {len(train_dataset)} training examples.')
print(f'\nSample formatted example:')
print('─' * 60)
print(train_dataset[0]['text'][:800])
print('─' * 60)

Formatting training data...


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

✅ Formatted 7473 training examples.

Sample formatted example:
────────────────────────────────────────────────────────────
Solve this math problem carefully.

First, identify what the problem is asking for.
Then solve step by step, tracking all intermediate values.
Finally, write your answer.

Problem: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

What I need to find: Step 1: Natalia sold 48/2 = 24 clips in May.
Step 2: Natalia sold 48+24 = 72 clips altogether in April and May.
Answer: 72
────────────────────────────────────────────────────────────


## Cell 4 — Load Model with LoRA

**Why LoRA instead of full fine-tuning:**
- Full fine-tuning Gemma 2B (2.5B params) would need ~40GB VRAM
- LoRA with r=16 adds only ~8M trainable parameters on top of frozen weights
- Fits comfortably in 16GB P100 VRAM
- Near-identical quality to full fine-tuning on domain-specific tasks

In [11]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 12.4 MB/s eta 0:00:0000:01


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f'Loading base model: {MODEL_ID}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Add padding token if missing (Gemma doesn't have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load in float16 — fits in 16GB, faster than float32
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

print(f'✅ Base model loaded.')
print(f'   Parameters: {sum(p.numel() for p in base_model.parameters())/1e9:.2f}B')
print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading base model: microsoft/Phi-3-mini-4k-instruct


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Base model loaded.
   Parameters: 3.82B
   VRAM used: 3.82 GB


In [13]:
# Apply LoRA adapter
lora_config = LoraConfig(
    r=LORA_CONFIG['r'],
    lora_alpha=LORA_CONFIG['lora_alpha'],
    target_modules=LORA_CONFIG['target_modules'],
    lora_dropout=LORA_CONFIG['lora_dropout'],
    bias=LORA_CONFIG['bias'],
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)

# Print trainable parameter count
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nLoRA adapter applied:')
print(f'  Total parameters:     {total_params/1e9:.3f}B')
print(f'  Trainable parameters: {train_params/1e6:.1f}M ({100*train_params/total_params:.2f}%)')
print(f'  Frozen parameters:    {(total_params-train_params)/1e9:.3f}B')
print(f'  VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Verify LoRA is attached
print(f'\nLoRA config: r={LORA_CONFIG["r"]}, alpha={LORA_CONFIG["lora_alpha"]}, dropout={LORA_CONFIG["lora_dropout"]}')


LoRA adapter applied:
  Total parameters:     3.839B
  Trainable parameters: 17.8M (0.46%)
  Frozen parameters:    3.821B
  VRAM used: 3.86 GB

LoRA config: r=32, alpha=32, dropout=0.05


## Cell 5 — Fine-Tune

**Expected runtime:** ~3-5 hours on P100 for 3 epochs over 7,473 examples.

**What to watch:** The training loss should drop from ~2.0 → ~0.8-1.0 over 3 epochs. If it's not dropping, the learning rate is too low. If it drops to <0.3 very fast, you're overfitting.

In [19]:
import json

sft_config = SFTConfig(
    max_steps=300,  
    output_dir=FINETUNED_MODEL_DIR,
    num_train_epochs=FINETUNE_CONFIG['num_train_epochs'],
    per_device_train_batch_size=FINETUNE_CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=FINETUNE_CONFIG['gradient_accumulation_steps'],
    learning_rate=FINETUNE_CONFIG['learning_rate'],
    warmup_ratio=FINETUNE_CONFIG['warmup_ratio'],
    lr_scheduler_type='cosine',       # Cosine decay — smoother than linear for reasoning
    fp16=True,                        # Mixed precision — required for float16 model
    logging_steps=FINETUNE_CONFIG['logging_steps'],
    save_strategy='steps',
    save_steps=FINETUNE_CONFIG['save_steps'],
    save_total_limit=2,               # Keep only last 2 checkpoints (save disk space)
    load_best_model_at_end=False,     # No eval set — we evaluate manually after
    report_to='none',                 # Disable wandb/tensorboard
    seed=RANDOM_SEED,
    dataset_text_field='text',
    packing=False,                    # Don't pack sequences — cleaner for instruction tuning
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=sft_config,
    processing_class=tokenizer,
)


print('Starting fine-tuning...')
print(f'  Epochs: {FINETUNE_CONFIG["num_train_epochs"]}')
print(f'  Effective batch size: {FINETUNE_CONFIG["per_device_train_batch_size"] * FINETUNE_CONFIG["gradient_accumulation_steps"]}')
print(f'  Learning rate: {FINETUNE_CONFIG["learning_rate"]}')
print(f'  Training examples: {len(train_dataset)}')
print(f'  Steps per epoch: {len(train_dataset) // (FINETUNE_CONFIG["per_device_train_batch_size"] * FINETUNE_CONFIG["gradient_accumulation_steps"])}')
print('─' * 60)

train_result = trainer.train()

print('\n' + '─' * 60)
print('✅ Fine-tuning complete.')
print(f'  Final training loss: {train_result.training_loss:.4f}')
print(f'  Total steps: {train_result.global_step}')
print(f'  Runtime: {train_result.metrics["train_runtime"]/3600:.1f} hours')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning...
  Epochs: 3
  Effective batch size: 16
  Learning rate: 0.0002
  Training examples: 7473
  Steps per epoch: 467
────────────────────────────────────────────────────────────


Step,Training Loss
20,0.235996
40,0.227121
60,0.234090
80,0.223896
100,0.225062
120,0.227328
140,0.225238
160,0.225742
180,0.219556
200,0.220873



────────────────────────────────────────────────────────────
✅ Fine-tuning complete.
  Final training loss: 0.2234
  Total steps: 300
  Runtime: 0.6 hours


## Cell 6 — Save Fine-Tuned Model

In [20]:
# Save the LoRA adapter weights
# These are small (~30MB) — the base model weights are unchanged and not saved
print('Saving fine-tuned model...')

trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)

# Save training config for reproducibility
with open(FINETUNED_MODEL_DIR + 'training_config.json', 'w') as f:
    json.dump({
        'base_model': MODEL_ID,
        'finetune_config': FINETUNE_CONFIG,
        'lora_config': {k: v for k, v in LORA_CONFIG.items() if k != 'target_modules'},
        'lora_target_modules': LORA_CONFIG['target_modules'],
        'final_loss': train_result.training_loss,
        'total_steps': train_result.global_step,
    }, f, indent=2)

print(f'✅ Saved to {FINETUNED_MODEL_DIR}')
print('\nFiles saved:')
for f in os.listdir(FINETUNED_MODEL_DIR):
    fpath = os.path.join(FINETUNED_MODEL_DIR, f)
    if os.path.isfile(fpath):
        print(f'  {f}: {os.path.getsize(fpath)/1e6:.1f} MB')

Saving fine-tuned model...
✅ Saved to /kaggle/working/gemma2b_gsm8k_lora/

Files saved:
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 71.3 MB
  tokenizer_config.json: 0.0 MB
  tokenizer.json: 3.6 MB
  README.md: 0.0 MB
  training_config.json: 0.0 MB
  training_args.bin: 0.0 MB
  chat_template.jinja: 0.0 MB


## Cell 7 — Reload Fine-Tuned Model for Evaluation

We merge LoRA weights back into the base model for cleaner inference.
This is optional but makes generation slightly faster.

In [21]:
from peft import PeftModel
import gc

# Free trainer memory
del trainer
torch.cuda.empty_cache()
gc.collect()

print('Merging LoRA weights into base model for inference...')

# Reload base model fresh
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

# Load LoRA adapter and merge
peft_model = PeftModel.from_pretrained(base_model_eval, FINETUNED_MODEL_DIR)
merged_model = peft_model.merge_and_unload()  # Merges LoRA into base weights
merged_model.eval()

# This is now our inference model — replace the variable name for clarity
model = merged_model

print(f'✅ Fine-tuned model ready for inference.')
print(f'   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Merging LoRA weights into base model for inference...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Fine-tuned model ready for inference.
   VRAM: 7.70 GB


---
# PHASE 2: SLD-VM EVALUATION PIPELINE
---

Everything below is the same SLD-VM pipeline from the fixed notebook, now running on the fine-tuned model. The fine-tuned model produces much better per-branch accuracy, which means:
- Structured CoT alone should hit ~28-32% (vs 14% before)
- Self-consistency will now genuinely help (branches agree more on correct answers)
- VCE has real signal to work with

## Cell 8 — Helper Functions

In [31]:
import re, json, pickle
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# GENERATION
# ═══════════════════════════════════════════════════════════

def generate_text(prompt, max_new_tokens=512, temperature=0.0, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if temperature > 0 else 1.0,
            top_p=top_p,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

# ═══════════════════════════════════════════════════════════
# ANSWER EXTRACTION — robust multi-pattern
# ═══════════════════════════════════════════════════════════

def clean_number(s):
    return s.strip().rstrip('.,;:').replace(',', '').replace('$', '').replace('%', '')

def extract_answer(text):
    if not text:
        return None
    patterns = [
        r'(?:Answer|answer|ANSWER)\s*[:=]\s*\$?([\-0-9,\.]+)',
        r'(?:The answer is|Therefore,?|Thus,?)\s*\$?([\-0-9,\.]+)',
        r'####\s*([\-0-9,\.]+)',
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            val = clean_number(m.group(1))
            if val:
                return val
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        if re.match(r'^\$?[\-0-9,\.]+$', line):
            val = clean_number(line)
            if val:
                return val
    last_text = ' '.join(lines[-3:]) if len(lines) >= 3 else text
    numbers = re.findall(r'-?\d+\.?\d*', last_text)
    return clean_number(numbers[-1]) if numbers else None

def get_ground_truth(answer_text):
    nums = re.findall(r'####\s*([\-0-9,\.]+)', answer_text)
    if nums:
        return clean_number(nums[-1])
    nums = re.findall(r'-?\d+\.?\d*', answer_text)
    return clean_number(nums[-1]) if nums else None

def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 0.01
    except:
        return pred.strip() == gold.strip()

# ═══════════════════════════════════════════════════════════
# ARITHMETIC VERIFICATION
# ═══════════════════════════════════════════════════════════

def verify_arithmetic(text):
    equations = re.findall(
        r'([\d\.\+\-\*/\(\)\s]+)\s*=\s*([\d\.\+\-\*/\(\)\s]+)', text
    )
    if not equations:
        return 0.5
    correct, total = 0, 0
    for left, right in equations:
        try:
            if abs(eval(left.strip()) - eval(right.strip())) <= 0.01:
                correct += 1
            total += 1
        except:
            continue
    return correct / total if total > 0 else 0.5

# ═══════════════════════════════════════════════════════════
# EMBEDDING (for PRM classifier)
# ═══════════════════════════════════════════════════════════

def get_embedding(text, max_length=128):
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_length, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states[-1]
    mask = inputs['attention_mask'].unsqueeze(-1).float()
    return ((hidden * mask).sum(1) / mask.sum(1)).squeeze(0).cpu().float().numpy()

print('✅ Helper functions loaded.')

✅ Helper functions loaded.


## Cell 9 — Load Test Data

In [32]:
N_EVAL = EVAL_CONFIG['n_eval']

# Use the already-loaded test split
gsm8k_eval = gsm8k_test.select(range(N_EVAL))
print(len(gsm8k_test))

print(f'✅ Evaluating on {N_EVAL} GSM8K test problems.')
print(f'\nExample:')
print(f'Q: {gsm8k_eval[0]["question"]}')
print(f'A: {gsm8k_eval[0]["answer"]}')

1319
✅ Evaluating on 200 GSM8K test problems.

Example:
Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
A: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


## Cell 10 — Stage 0: Zero-Shot Baseline (Fine-Tuned Model)

In [33]:
BASELINE_PROMPT = """Solve this math problem. Give only the final numeric answer.

Problem: {problem}

Answer:"""

results_baseline = []
print(f'Running zero-shot baseline on fine-tuned model ({N_EVAL} problems)...')

for i, example in enumerate(tqdm(gsm8k_eval, desc='FT Baseline')):
    problem = example['question']
    correct_answer = get_ground_truth(example['answer'])
    generated = generate_text(BASELINE_PROMPT.format(problem=problem), temperature=0.0)
    pred_answer = extract_answer(generated)
    is_correct = answers_match(pred_answer, correct_answer)
    results_baseline.append({
        'problem_id': i, 'problem': problem, 'generated': generated,
        'pred_answer': pred_answer, 'correct_answer': correct_answer, 'is_correct': is_correct,
    })
    if (i + 1) % 50 == 0:
        acc = sum(r['is_correct'] for r in results_baseline) / len(results_baseline)
        print(f'  [{i+1}/{N_EVAL}] Accuracy: {acc:.1%}')

ft_baseline_acc = sum(r['is_correct'] for r in results_baseline) / len(results_baseline)
print(f'\n{"="*60}')
print(f'FT Zero-Shot Baseline: {ft_baseline_acc:.1%}')
print(f'{"="*60}')
pd.DataFrame(results_baseline).to_csv(EVAL_OUTPUT_DIR + 'stage0_ft_baseline.csv', index=False)
print('✅ Saved.')

Running zero-shot baseline on fine-tuned model (200 problems)...


FT Baseline:   0%|          | 0/200 [00:00<?, ?it/s]

  [50/200] Accuracy: 72.0%
  [100/200] Accuracy: 79.0%
  [150/200] Accuracy: 79.3%
  [200/200] Accuracy: 78.5%

FT Zero-Shot Baseline: 78.5%
✅ Saved.


## Cell 11 — Stage 1: Structured CoT (Fine-Tuned Model)

This is where the big gain comes from. The fine-tuned model has been trained to complete exactly this prompt format. It learned from 7,473 examples of step-by-step reasoning — it now knows how GSM8K problems are structured.

In [34]:
# This prompt MUST match the training format from Cell 3 exactly
# The model learned to complete this specific template
STRUCTURED_COT_PROMPT = """Solve this math problem carefully.

First, identify what the problem is asking for.
Then solve step by step, tracking all intermediate values.
Finally, write your answer.

Problem: {problem}

What I need to find: """

results_scot = []
print(f'Running Structured CoT on fine-tuned model ({N_EVAL} problems)...')

for i, example in enumerate(tqdm(gsm8k_eval, desc='FT Structured CoT')):
    problem = example['question']
    correct_answer = get_ground_truth(example['answer'])
    prompt = STRUCTURED_COT_PROMPT.format(problem=problem)
    generated = generate_text(prompt, temperature=0.0, max_new_tokens=EVAL_CONFIG['max_new_tokens'])
    pred_answer = extract_answer(generated)
    is_correct = answers_match(pred_answer, correct_answer)
    results_scot.append({
        'problem_id': i, 'problem': problem, 'generated': generated,
        'pred_answer': pred_answer, 'correct_answer': correct_answer, 'is_correct': is_correct,
    })
    if (i + 1) % 50 == 0:
        acc = sum(r['is_correct'] for r in results_scot) / len(results_scot)
        print(f'  [{i+1}/{N_EVAL}] Accuracy: {acc:.1%}')

scot_acc = sum(r['is_correct'] for r in results_scot) / len(results_scot)
print(f'\n{"="*60}')
print(f'FT Structured CoT: {scot_acc:.1%}')
print(f'Δ vs FT zero-shot: {scot_acc - ft_baseline_acc:+.1%}')
print(f'{"="*60}')
# Expected: ~28-32% at this stage
pd.DataFrame(results_scot).to_csv(EVAL_OUTPUT_DIR + 'stage1_ft_structured_cot.csv', index=False)
print('✅ Saved.')

Running Structured CoT on fine-tuned model (200 problems)...


FT Structured CoT:   0%|          | 0/200 [00:00<?, ?it/s]

  [50/200] Accuracy: 70.0%
  [100/200] Accuracy: 72.0%
  [150/200] Accuracy: 69.3%
  [200/200] Accuracy: 69.5%

FT Structured CoT: 69.5%
Δ vs FT zero-shot: -9.0%
✅ Saved.


## Cell 12 — Train Proxy PRM on Fine-Tuned Model Outputs

We retrain the PRM on the fine-tuned model's CoT outputs.
Do NOT reuse the PRM trained on the base model — the embedding space has shifted after fine-tuning.

In [35]:
USE_PRM = False
prm_clf = None

print('Training Proxy PRM on fine-tuned model CoT outputs...')

solution_texts = [r['generated'] for r in results_scot]
solution_labels = np.array([1 if r['is_correct'] else 0 for r in results_scot])

n_pos = solution_labels.sum()
n_neg = (solution_labels == 0).sum()
print(f'Labels: {n_pos} correct, {n_neg} incorrect ({n_pos/(n_pos+n_neg):.1%} positive rate)')

# Need at least 10 positives to train a meaningful classifier
if n_pos < 10:
    print('⚠️  Too few positive examples. Falling back to majority vote in VCE.')
else:
    pos_idx = np.where(solution_labels == 1)[0]
    neg_idx = np.where(solution_labels == 0)[0]
    n_each = min(len(pos_idx), len(neg_idx))

    rng = np.random.RandomState(RANDOM_SEED)
    balanced_idx = np.concatenate([
        rng.choice(pos_idx, n_each, replace=False),
        rng.choice(neg_idx, n_each, replace=False),
    ])

    print(f'Training on {len(balanced_idx)} balanced examples ({n_each} per class).')
    print('Extracting embeddings (this takes ~10-15 mins)...')

    X = np.array([
        get_embedding(solution_texts[i])
        for i in tqdm(balanced_idx, desc='Embeddings')
    ])
    y = solution_labels[balanced_idx]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
    )

    prm_clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0)
    prm_clf.fit(X_train, y_train)

    print(f'\nProxy PRM — Train acc: {prm_clf.score(X_train, y_train):.1%} | Val acc: {prm_clf.score(X_test, y_test):.1%}')
    print('Note: Higher val acc = VCE will make better branch selection decisions')

    with open(EVAL_OUTPUT_DIR + 'ft_proxy_prm.pkl', 'wb') as f:
        pickle.dump(prm_clf, f)

    USE_PRM = True
    print('✅ PRM trained and saved.')

print(f'\nPRM status: {"Enabled" if USE_PRM else "Disabled — VCE uses majority vote"}')

Training Proxy PRM on fine-tuned model CoT outputs...
Labels: 139 correct, 61 incorrect (69.5% positive rate)
Training on 122 balanced examples (61 per class).
Extracting embeddings (this takes ~10-15 mins)...


Embeddings:   0%|          | 0/122 [00:00<?, ?it/s]


Proxy PRM — Train acc: 100.0% | Val acc: 68.0%
Note: Higher val acc = VCE will make better branch selection decisions
✅ PRM trained and saved.

PRM status: Enabled


## Cell 13 — Stage 2: Self-Consistency RGC

With the fine-tuned model, self-consistency finally works as intended:
- Per-branch accuracy ~28-32% means the correct answer appears in 1 of 3 branches more often
- Majority vote selects the correct answer more often than any single branch
- Agreement rate will be higher (model is more confident and consistent)

In [36]:
def majority_vote(answers):
    valid = [a for a in answers if a is not None]
    if not valid:
        return None
    groups = {}
    for a in valid:
        try:
            key = round(float(a), 2)
        except:
            key = a
        groups.setdefault(key, [])
        groups[key].append(a)
    best_key = max(groups, key=lambda k: len(groups[k]))
    return groups[best_key][0]

results_rgc = []
N_BRANCHES = EVAL_CONFIG['n_branches']
TEMPS = EVAL_CONFIG['branch_temperatures']

print(f'Running Self-Consistency RGC ({N_BRANCHES} branches, temps={TEMPS})...')
print(f'Estimated time: ~{N_EVAL * N_BRANCHES * 5 / 60:.0f} minutes\n')

for i, example in enumerate(tqdm(gsm8k_eval, desc='Self-Consistency RGC')):
    problem = example['question']
    correct_answer = get_ground_truth(example['answer'])

    branches = []
    branch_answers = []

    for temp in TEMPS:
        prompt = STRUCTURED_COT_PROMPT.format(problem=problem)
        generated = generate_text(prompt, temperature=temp, max_new_tokens=EVAL_CONFIG['max_new_tokens'])
        answer = extract_answer(generated)
        branches.append({'generated': generated, 'answer': answer, 'temperature': temp})
        branch_answers.append(answer)

    final_answer = majority_vote(branch_answers)
    is_correct = answers_match(final_answer, correct_answer)

    results_rgc.append({
        'problem_id': i, 'problem': problem,
        'branches': branches,
        'branch_answers': branch_answers,
        'majority_vote': final_answer,
        'correct_answer': correct_answer,
        'is_correct': is_correct,
    })

    if (i + 1) % 50 == 0:
        acc = sum(r['is_correct'] for r in results_rgc) / len(results_rgc)
        agree = sum(
            1 for r in results_rgc
            if len(set(str(a) for a in r['branch_answers'] if a)) == 1
        )
        print(f'  [{i+1}/{N_EVAL}] Accuracy: {acc:.1%} | Full agreement: {agree/len(results_rgc):.1%}')

rgc_acc = sum(r['is_correct'] for r in results_rgc) / len(results_rgc)
print(f'\n{"="*60}')
print(f'Self-Consistency RGC: {rgc_acc:.1%}')
print(f'Δ vs Structured CoT: {rgc_acc - scot_acc:+.1%}')
print(f'{"="*60}')

df_rgc = pd.DataFrame([{
    'problem_id': r['problem_id'], 'problem': r['problem'],
    'branch_answers': json.dumps(r['branch_answers']),
    'majority_vote': r['majority_vote'],
    'correct_answer': r['correct_answer'],
    'is_correct': r['is_correct'],
} for r in results_rgc])
df_rgc.to_csv(EVAL_OUTPUT_DIR + 'stage2_rgc.csv', index=False)
print('✅ Saved.')

Running Self-Consistency RGC (3 branches, temps=[0.3, 0.6, 0.9])...
Estimated time: ~50 minutes



Self-Consistency RGC:   0%|          | 0/200 [00:00<?, ?it/s]

  [50/200] Accuracy: 70.0% | Full agreement: 34.0%
  [100/200] Accuracy: 76.0% | Full agreement: 38.0%
  [150/200] Accuracy: 70.0% | Full agreement: 36.0%
  [200/200] Accuracy: 70.0% | Full agreement: 36.0%

Self-Consistency RGC: 70.0%
Δ vs Structured CoT: +0.5%
✅ Saved.


## Cell 14 — Stage 3: VCE (Final Pipeline)

In [37]:
def vce_score_branch(branch_text):
    if USE_PRM and prm_clf is not None:
        try:
            emb = get_embedding(branch_text)
            prm_score = prm_clf.predict_proba(emb.reshape(1, -1))[0][1]
        except:
            prm_score = 0.5
    else:
        prm_score = 0.5
    rule_score = verify_arithmetic(branch_text)
    return (
        EVAL_CONFIG['vce_prm_weight'] * prm_score +
        EVAL_CONFIG['vce_rule_weight'] * rule_score
    )

def vce_select(branches, majority_answer):
    if not branches:
        return majority_answer
    scored = [(vce_score_branch(b['generated']), b['answer']) for b in branches]
    scored.sort(reverse=True, key=lambda x: x[0])
    # Only override majority if VCE has clear confidence advantage
    if len(scored) > 1 and (scored[0][0] - scored[1][0]) < 0.05:
        return majority_answer
    return scored[0][1]

results_vce = []
print(f'Running VCE ({N_EVAL} problems)...')
print(f'PRM={USE_PRM} | Weights: {EVAL_CONFIG["vce_prm_weight"]}/{EVAL_CONFIG["vce_rule_weight"]}\n')

for i, rgc_result in enumerate(tqdm(results_rgc, desc='VCE')):
    branches = rgc_result['branches']
    majority_answer = rgc_result['majority_vote']
    correct_answer = rgc_result['correct_answer']

    vce_answer = vce_select(branches, majority_answer)
    is_correct = answers_match(vce_answer, correct_answer)

    results_vce.append({
        'problem_id': i, 'problem': rgc_result['problem'],
        'vce_answer': vce_answer,
        'majority_answer': majority_answer,
        'correct_answer': correct_answer,
        'is_correct': is_correct,
        'differs_from_majority': str(vce_answer) != str(majority_answer),
    })

    if (i + 1) % 50 == 0:
        acc = sum(r['is_correct'] for r in results_vce) / len(results_vce)
        print(f'  [{i+1}/{N_EVAL}] Accuracy: {acc:.1%}')

vce_acc = sum(r['is_correct'] for r in results_vce) / len(results_vce)
print(f'\n{"="*60}')
print(f'VCE Full Pipeline: {vce_acc:.1%}')
print(f'Δ vs RGC: {vce_acc - rgc_acc:+.1%}')
print(f'{"="*60}')
pd.DataFrame(results_vce).to_csv(EVAL_OUTPUT_DIR + 'stage3_vce.csv', index=False)
print('✅ Saved.')

Running VCE (200 problems)...
PRM=True | Weights: 0.7/0.3



VCE:   0%|          | 0/200 [00:00<?, ?it/s]

  [50/200] Accuracy: 60.0%
  [100/200] Accuracy: 71.0%
  [150/200] Accuracy: 67.3%
  [200/200] Accuracy: 68.5%

VCE Full Pipeline: 68.5%
Δ vs RGC: -1.5%
✅ Saved.


## Cell 15 — Final Results

In [38]:
# ═══════════════════════════════════════════════════════════
# COMPLETE ABLATION TABLE
# Shows full picture: base model vs fine-tuned at each stage
# ═══════════════════════════════════════════════════════════

print('\n' + '='*75)
print('FINAL RESULTS — Fine-Tuned Gemma 2B + SLD-VM Pipeline')
print('='*75)

# Reference numbers from the base model run (hardcode from previous notebook)
# Update these with your actual base model results
BASE_COT = 0.125        # Your CoT baseline (12.5%)
BASE_SCOT = 0.140       # Your structured CoT (14.0%)
BASE_PIPELINE = 0.100   # Your full pipeline (10.0%)

ablation = {
    'Configuration': [
        'Base Gemma 2B (CoT baseline)',
        'Base Gemma 2B (Structured CoT)',
        'Base Gemma 2B (Full Pipeline)',
        '─────────────────────────────',
        'Fine-Tuned 2B (zero-shot)',
        'Fine-Tuned 2B + Structured CoT',
        'Fine-Tuned 2B + Self-Consistency',
        'Fine-Tuned 2B + Full SLD-VM Pipeline',
    ],
    'Accuracy': [
        f'{BASE_COT:.1%}',
        f'{BASE_SCOT:.1%}',
        f'{BASE_PIPELINE:.1%}',
        '─────────',
        f'{ft_baseline_acc:.1%}',
        f'{scot_acc:.1%}',
        f'{rgc_acc:.1%}',
        f'{vce_acc:.1%}',
    ],
    'Δ vs Base CoT': [
        '0.0%',
        f'{BASE_SCOT - BASE_COT:+.1%}',
        f'{BASE_PIPELINE - BASE_COT:+.1%}',
        '─────────',
        f'{ft_baseline_acc - BASE_COT:+.1%}',
        f'{scot_acc - BASE_COT:+.1%}',
        f'{rgc_acc - BASE_COT:+.1%}',
        f'{vce_acc - BASE_COT:+.1%}',
    ],
}

df_final = pd.DataFrame(ablation)
print(df_final.to_string(index=False))
print('='*75)

# ═══════════════════════════════════════════════════════════
# PIPELINE COMPONENT CONTRIBUTION (fine-tuned model)
# ═══════════════════════════════════════════════════════════

print(f'\n{"═"*75}')
print('COMPONENT CONTRIBUTION (Fine-Tuned Model)')
print(f'{"═"*75}')
print(f'LoRA Fine-Tuning gain:          {scot_acc - BASE_SCOT:+.1%}  (structured CoT: FT vs base)')
print(f'Self-Consistency gain:          {rgc_acc - scot_acc:+.1%}  (vs structured CoT alone)')
print(f'VCE gain:                       {vce_acc - rgc_acc:+.1%}  (vs self-consistency)')
print(f'─'*45)
print(f'Total gain vs base CoT:         {vce_acc - BASE_COT:+.1%}')
print(f'Target range was:               +22% to +26%')
print(f'Target met: {"✅ YES" if vce_acc >= 0.34 else "❌ Not quite — see notes below"}')
print(f'{"═"*75}')

# ═══════════════════════════════════════════════════════════
# VCE IMPACT
# ═══════════════════════════════════════════════════════════

vce_improvements = sum(
    1 for i in range(len(results_vce))
    if results_vce[i]['is_correct'] and not results_rgc[i]['is_correct']
)
vce_regressions = sum(
    1 for i in range(len(results_vce))
    if not results_vce[i]['is_correct'] and results_rgc[i]['is_correct']
)
print(f'\nVCE improvements: {vce_improvements}')
print(f'VCE regressions:  {vce_regressions}')
print(f'Net VCE benefit:  {vce_improvements - vce_regressions:+d} problems')

# ═══════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════

df_final.to_csv(EVAL_OUTPUT_DIR + 'final_ablation_table.csv', index=False)

summary = {
    'model': MODEL_ID,
    'finetuned': True,
    'n_eval': N_EVAL,
    'results': {
        'base_cot': BASE_COT,
        'base_structured_cot': BASE_SCOT,
        'base_pipeline': BASE_PIPELINE,
        'ft_zeroshot': ft_baseline_acc,
        'ft_structured_cot': scot_acc,
        'ft_self_consistency': rgc_acc,
        'ft_full_pipeline': vce_acc,
    },
    'config': EVAL_CONFIG,
    'lora_config': LORA_CONFIG,
    'finetune_config': FINETUNE_CONFIG,
}
with open(EVAL_OUTPUT_DIR + 'experiment_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n✅ All results saved to {EVAL_OUTPUT_DIR}')
print('\nFiles:')
for fname in sorted(os.listdir(EVAL_OUTPUT_DIR)):
    fpath = os.path.join(EVAL_OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  {fname:40s} ({os.path.getsize(fpath)/1024:>7.1f} KB)')


FINAL RESULTS — Fine-Tuned Gemma 2B + SLD-VM Pipeline
                       Configuration  Accuracy Δ vs Base CoT
        Base Gemma 2B (CoT baseline)     12.5%          0.0%
      Base Gemma 2B (Structured CoT)     14.0%         +1.5%
       Base Gemma 2B (Full Pipeline)     10.0%         -2.5%
       ───────────────────────────── ─────────     ─────────
           Fine-Tuned 2B (zero-shot)     78.5%        +66.0%
      Fine-Tuned 2B + Structured CoT     69.5%        +57.0%
    Fine-Tuned 2B + Self-Consistency     70.0%        +57.5%
Fine-Tuned 2B + Full SLD-VM Pipeline     68.5%        +56.0%

═══════════════════════════════════════════════════════════════════════════
COMPONENT CONTRIBUTION (Fine-Tuned Model)
═══════════════════════════════════════════════════════════════════════════
LoRA Fine-Tuning gain:          +55.5%  (structured CoT: FT vs base)
Self-Consistency gain:          +0.5%  (vs structured CoT alone)
VCE gain:                       -1.5%  (vs self-consistency)
──────

## Cell 16 — Package & Download

In [39]:
import shutil

# Package evaluation results
shutil.make_archive('/kaggle/working/sldvm_finetuned_results', 'zip', EVAL_OUTPUT_DIR)
print('✅ Evaluation results packaged: sldvm_finetuned_results.zip')

# Note: Fine-tuned model adapter is in FINETUNED_MODEL_DIR
# It's ~30-50MB — you can download it separately if you want to reuse it
print(f'\nFine-tuned LoRA adapter: {FINETUNED_MODEL_DIR}')
print('📥 Download from Kaggle output panel →')

print('\n' + '='*60)
print('DONE. Summary of what was produced:')
print('='*60)
print(f'1. LoRA adapter saved to:   {FINETUNED_MODEL_DIR}')
print(f'2. Evaluation results in:   {EVAL_OUTPUT_DIR}')
print(f'3. Final accuracy achieved: {vce_acc:.1%}')
print('='*60)

✅ Evaluation results packaged: sldvm_finetuned_results.zip

Fine-tuned LoRA adapter: /kaggle/working/gemma2b_gsm8k_lora/
📥 Download from Kaggle output panel →

DONE. Summary of what was produced:
1. LoRA adapter saved to:   /kaggle/working/gemma2b_gsm8k_lora/
2. Evaluation results in:   /kaggle/working/sldvm_finetuned_eval/
3. Final accuracy achieved: 68.5%


In [40]:
import shutil

# Zip the model — Kaggle can only download single files, not folders
shutil.make_archive('/kaggle/working/phi3_finetuned_model', 'zip', '/kaggle/working/gemma2b_gsm8k_lora/')
shutil.make_archive('/kaggle/working/sldvm_eval_results', 'zip', '/kaggle/working/sldvm_finetuned_eval/')

print("✅ Files ready to download:")
print("  - phi3_finetuned_model.zip  (LoRA adapter weights)")
print("  - sldvm_eval_results.zip    (all CSVs and summary)")

✅ Files ready to download:
  - phi3_finetuned_model.zip  (LoRA adapter weights)
  - sldvm_eval_results.zip    (all CSVs and summary)


---
## Notes for Troubleshooting

**If fine-tuning loss doesn't drop below 1.5:**
Increase learning rate to `3e-4` or increase LoRA rank to `r=32`.

**If VRAM OOM during fine-tuning:**
Reduce `per_device_train_batch_size` to 1 and increase `gradient_accumulation_steps` to 16.

**If self-consistency hurts accuracy (like the base model):**
Your fine-tuned model's per-sample accuracy is still too low. Check Stage 1 (Structured CoT). If it's below 20%, train for more epochs or check that training data formatting matched the inference prompt.

**If you hit ~30-32% but not 34-38%:**
Increase branches to 5 (`n_branches: 5`, temps `[0.2, 0.4, 0.6, 0.8, 1.0]`). More votes = better majority. Runtime goes from 2hr → 3.5hr for eval.

**For a paper:** Run on full 1,319 test problems (`n_eval: 1319`). Dev set numbers will be slightly optimistic.